# 31 — Resource Allocation Determines Quantum Utility

**Engineering question**

Given admissible quantum photonic resources, where should engineering effort be spent?

Notebook 29 established that loss and related constraints determine how much of an ideal cluster state remains usable. Notebook 31 begins Part II by asking how an engineer should allocate effort across the remaining admissible resource.

```text
generated resources
→ admissibility tests
→ directed allocation
→ critical paths
→ logical computation
→ measurement
→ refinement
```

Engineering statement:

> Engineering decisions determine which resources remain admissible for useful quantum computation. Resource allocation specifies progression through the computational pipeline.


## Setup

This notebook creates:

```text
figures/
results/csv/
results/json/
results/31_outputs.zip
```


In [ ]:
from pathlib import Path
import json, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIG = ROOT / "figures"
RES = ROOT / "results"
CSV = RES / "csv"
JS = RES / "json"

for p in [FIG, RES, CSV, JS]:
    p.mkdir(parents=True, exist_ok=True)

def savefig(fig, filename, dpi=220):
    path = FIG / filename
    fig.tight_layout()
    fig.savefig(path, dpi=dpi)
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)
print("FIG:", FIG)
print("RES:", RES)


## 1. Uniform allocation versus directed allocation

A uniform allocation treats every graph edge equally. A directed allocation asks which edges and nodes matter most for the computation.

```text
resources should follow computational importance,
not merely geometric symmetry.
```


In [ ]:
rows, cols = 4, 6
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))

for ax, title, directed in zip(axes, ["Uniform allocation", "Directed allocation"], [False, True]):
    ax.set_title(title, fontsize=14)
    ax.axis("off")
    ax.set_aspect("equal")
    xs = np.arange(cols)
    ys = np.arange(rows)
    for y in ys:
        for x in xs[:-1]:
            lw = 2.0
            if directed and y == 2 and x in [1, 2, 3]:
                lw = 4.2
            elif directed:
                lw = 1.1
            ax.plot([x, x + 1], [y, y], linewidth=lw, alpha=0.95 if lw > 2 else 0.45)
    for x in xs:
        for y in ys[:-1]:
            lw = 2.0
            if directed and x == 3 and y in [1, 2]:
                lw = 4.2
            elif directed:
                lw = 1.1
            ax.plot([x, x], [y, y + 1], linewidth=lw, alpha=0.95 if lw > 2 else 0.45)
    for y in ys:
        for x in xs:
            size = 210 if directed and ((y == 2 and x in [1,2,3,4]) or (x == 3 and y in [1,2,3])) else 120
            ax.scatter([x], [y], s=size)
    subtitle = "equal squeezing / routing / measurement" if not directed else "priority path receives stronger admissibility budget"
    ax.text((cols-1)/2, -0.55, subtitle, ha="center", fontsize=11)

fig.suptitle("Engineering Allocation Determines Computational Utility", fontsize=16)
savefig(fig, "31_uniform_vs_directed_allocation.png")
plt.show()


## 2. Admissibility pipeline

Part I asked how resources are generated. Part II asks whether they remain admissible for the next engineering stage.

Every arrow is a specification test.


In [ ]:
labels = [
    "Generated\nmodes",
    "Admissible\nfrequency modes",
    "Admissible\ngraph edges",
    "Admissible\nmeasurement paths",
    "Admissible\nfeed-forward",
    "Admissible\nlogical operations",
]
x = np.arange(len(labels))
fig, ax = plt.subplots(figsize=(13, 3.2))
for xi, label in zip(x, labels):
    rect = plt.Rectangle((xi - 0.46, -0.27), 0.92, 0.54, fill=False, linewidth=2)
    ax.add_patch(rect)
    ax.text(xi, 0, label, ha="center", va="center", fontsize=9.5)
for i in range(len(labels) - 1):
    ax.annotate("", xy=(i + 0.63, 0), xytext=(i + 0.46, 0), arrowprops=dict(arrowstyle="->", linewidth=2))
    ax.text(i + 0.55, 0.33, "test", ha="center", fontsize=8)
ax.set_title("Admissibility Pipeline: Each Stage Specifies the Next", fontsize=16)
ax.set_xlim(-0.75, len(labels) - 0.25)
ax.set_ylim(-0.7, 0.75)
ax.axis("off")
savefig(fig, "31_admissibility_pipeline.png")
plt.show()


## 3. Resource heat map

Not every node in a cluster graph contributes equally to a target computation. A heat map lets us distinguish critical nodes, important supporting nodes, and replaceable nodes.


In [ ]:
rows, cols = 6, 8
xv, yv = np.meshgrid(np.arange(cols), np.arange(rows))
importance = (
    np.exp(-((yv - 3.0)**2) / 1.6) * np.exp(-((xv - 4.2)**2) / 7.0)
    + 0.55 * np.exp(-((xv - 2.0)**2 + (yv - 1.5)**2) / 2.5)
)
importance = importance / importance.max()
fig, ax = plt.subplots(figsize=(9, 5.5))
im = ax.imshow(importance, origin="lower", vmin=0, vmax=1)
for y in range(rows):
    for x in range(cols - 1):
        ax.plot([x, x + 1], [y, y], linewidth=1.1, alpha=0.35)
for x in range(cols):
    for y in range(rows - 1):
        ax.plot([x, x], [y, y + 1], linewidth=1.1, alpha=0.35)
ax.scatter(xv.ravel(), yv.ravel(), s=90, edgecolors="black", linewidths=0.4)
ax.set_title("Resource Heat Map: Engineering Priority Is Not Uniform", fontsize=16)
ax.set_xlabel("Graph coordinate")
ax.set_ylabel("Graph coordinate")
ax.set_xticks(np.arange(cols))
ax.set_yticks(np.arange(rows))
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("relative allocation priority")
savefig(fig, "31_resource_heatmap.png")
plt.show()


## 4. Allocation budget

An engineering budget must be distributed. The exact percentages are illustrative; the point is that allocation is itself a design choice.


In [ ]:
categories = ["Squeezing", "Routing", "Detection", "Control", "Calibration", "Stabilization"]
budget = np.array([26, 19, 18, 16, 11, 10])
fig, ax = plt.subplots(figsize=(8.5, 5.2))
ax.pie(budget, labels=categories, autopct="%1.0f%%", startangle=90)
ax.set_title("Allocation Budget: Engineering Effort Is a Design Variable", fontsize=16)
savefig(fig, "31_allocation_budget.png")
plt.show()


## 5. Critical path

A cluster graph can be large, but a computation may depend on a smaller critical path.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.axis("off")
nodes = {
    "Pump": (0.08, 0.55),
    "Comb": (0.22, 0.55),
    "Graph A": (0.38, 0.78),
    "Graph B": (0.38, 0.55),
    "Graph C": (0.38, 0.32),
    "Measure A": (0.56, 0.78),
    "Measure B": (0.56, 0.55),
    "Measure C": (0.56, 0.32),
    "Feed-forward": (0.74, 0.55),
    "Logical output": (0.90, 0.55),
}
edges = [
    ("Pump", "Comb"), ("Comb", "Graph A"), ("Comb", "Graph B"), ("Comb", "Graph C"),
    ("Graph A", "Measure A"), ("Graph B", "Measure B"), ("Graph C", "Measure C"),
    ("Measure A", "Feed-forward"), ("Measure B", "Feed-forward"), ("Measure C", "Feed-forward"),
    ("Feed-forward", "Logical output"),
]
critical = {("Pump", "Comb"), ("Comb", "Graph B"), ("Graph B", "Measure B"), ("Measure B", "Feed-forward"), ("Feed-forward", "Logical output")}
for a, b in edges:
    lw = 4.0 if (a, b) in critical else 1.5
    alpha = 0.95 if (a, b) in critical else 0.35
    ax.annotate("", xy=nodes[b], xytext=nodes[a], arrowprops=dict(arrowstyle="->", linewidth=lw, alpha=alpha))
for name, (x, y) in nodes.items():
    size = 220 if name in ["Pump", "Comb", "Graph B", "Measure B", "Feed-forward", "Logical output"] else 130
    ax.scatter([x], [y], s=size, zorder=3)
    ax.text(x, y - 0.075, name, ha="center", fontsize=9)
ax.text(0.5, 0.08, "critical path determines admitted computation", ha="center", fontsize=12)
ax.set_title("Critical Path: Only Some Resources Determine Computation", fontsize=16)
ax.set_xlim(0, 1)
ax.set_ylim(0.05, 0.92)
savefig(fig, "31_critical_path.png")
plt.show()


## 6. Admissibility matrix

A resource is admitted or rejected by different specifications depending on the stage.


In [ ]:
rows = ["Squeezing", "Optical loss", "Mode control", "Detector efficiency", "Feed-forward"]
cols = ["Frequency mode", "Graph edge", "Measurement", "Logical qubit", "Algorithm depth"]
status = np.array([
    [1, 1, 0.5, 0.5, 0.0],
    [1, 0.5, 0.5, 0.0, 0.0],
    [1, 1, 0.5, 0.5, 0.0],
    [0.5, 0.5, 1, 0.5, 0.0],
    [0.0, 0.0, 0.5, 1, 0.5],
])
symbols = {1: "✓", 0.5: "•", 0: "×"}
fig, ax = plt.subplots(figsize=(9, 5.4))
im = ax.imshow(status, vmin=0, vmax=1)
ax.set_title("Admissibility Matrix Across Engineering Stages", fontsize=16)
ax.set_xticks(np.arange(len(cols)))
ax.set_yticks(np.arange(len(rows)))
ax.set_xticklabels(cols, rotation=25, ha="right")
ax.set_yticklabels(rows)
for i in range(status.shape[0]):
    for j in range(status.shape[1]):
        ax.text(j, i, symbols[float(status[i, j])], ha="center", va="center", fontsize=18)
cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("admissibility level")
savefig(fig, "31_admissibility_matrix.png")
plt.show()


## 7. Engineering summary

Notebook 31 reframes the problem:

```text
not merely: how many resources exist?
but: where do admissible resources produce computational value?
```


In [ ]:
summary = pd.DataFrame(
    [
        {"Resource": "Squeezing", "Allocation goal": "Preserve graph edges", "Leading specification": "Admissible correlations", "Result": "Graph utility"},
        {"Resource": "Routing", "Allocation goal": "Preserve topology", "Leading specification": "Admissible connectivity", "Result": "Cluster quality"},
        {"Resource": "Measurement", "Allocation goal": "Maximize readout information", "Leading specification": "Admissible readout", "Result": "Logical update"},
        {"Resource": "Feed-forward", "Allocation goal": "Minimize delay", "Leading specification": "Admissible timing", "Result": "Computation"},
        {"Resource": "Stabilization", "Allocation goal": "Preserve calibration", "Leading specification": "Admissible operation", "Result": "Repeatability"},
    ]
)
fig, ax = plt.subplots(figsize=(13, 3.8))
ax.axis("off")
table = ax.table(cellText=summary.values, colLabels=summary.columns, loc="center", cellLoc="center")
table.auto_set_font_size(False)
table.set_fontsize(9.4)
table.scale(1.08, 1.72)
for (r, c), cell in table.get_celld().items():
    cell.set_linewidth(1.2)
    if r == 0:
        cell.set_text_props(weight="bold")
ax.set_title("Engineering Summary: Allocation Follows Admissibility", fontsize=16, pad=20)
savefig(fig, "31_engineering_summary.png")
summary.to_csv(CSV / "31_summary.csv", index=False)
summary.to_json(JS / "31_summary.json", orient="records", indent=2)
plt.show()
summary


## 8. Closed design loop

Part I mostly moved forward:

```text
physics → resource → computation
```

Part II introduces iteration:

```text
computation → measurement → refinement → better allocation
```


In [ ]:
labels = ["Physics", "Resources", "Admissibility", "Engineering\nallocation", "Logical\ncomputation", "Measurement", "Refinement"]
theta = np.linspace(0, 2*np.pi, len(labels), endpoint=False)
xs = np.cos(theta)
ys = np.sin(theta)
fig, ax = plt.subplots(figsize=(7.5, 7.0))
ax.set_aspect("equal")
ax.axis("off")
for i in range(len(labels)):
    j = (i + 1) % len(labels)
    ax.annotate("", xy=(xs[j], ys[j]), xytext=(xs[i], ys[i]), arrowprops=dict(arrowstyle="->", linewidth=1.8))
ax.scatter(xs, ys, s=210, zorder=3)
for x, y, label in zip(xs, ys, labels):
    ax.text(1.22*x, 1.22*y, label, ha="center", va="center", fontsize=10)
ax.text(0, 0, "design\nloop", ha="center", va="center", fontsize=14)
ax.set_title("Complete Design Loop: Allocation Improves by Measurement", fontsize=16)
savefig(fig, "31_design_loop.png")
plt.show()


## 9. Export and download

This cell packages all outputs and starts a download.


In [ ]:
zip_path = RES / "31_outputs.zip"
files_to_zip = [
    FIG / "31_uniform_vs_directed_allocation.png",
    FIG / "31_admissibility_pipeline.png",
    FIG / "31_resource_heatmap.png",
    FIG / "31_allocation_budget.png",
    FIG / "31_critical_path.png",
    FIG / "31_admissibility_matrix.png",
    FIG / "31_engineering_summary.png",
    FIG / "31_design_loop.png",
    CSV / "31_summary.csv",
    JS / "31_summary.json",
]
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for file in files_to_zip:
        if file.exists():
            z.write(file, file.relative_to(ROOT))
print(f"Created: {zip_path}")
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    from IPython.display import FileLink, display
    display(FileLink(str(zip_path)))
summary


## Takeaways

- Resource count is not the same as resource utility.
- Admissibility specifies which resources can progress to the next engineering stage.
- Allocation should follow computational importance, not uniform graph geometry.
- Critical paths determine admitted computation.
- Measurement closes the design loop by informing the next allocation strategy.

## Next notebook

**37 — Programmable Photonic Architectures**

Notebook 37 should ask how programmable hardware makes allocation possible:

```text
admissible resource
→ programmable routing
→ tunable coupling
→ graph compilation
→ architecture-specific computation
```
